# SAT Encodings
## Full Adder

In order to be compliant to SAT, encoding numbers as binary values given a certain encoding length is the most viable way to represent integers. Moreover, mathematical functions as the sum should be adapeted to the new representaion as well.

This exercise session is devoted to the imlepmentation of the sum using a full adder.

In [ ]:
!pip3 install z3-solver

In [1]:
from z3 import *
from utils import decimal_to_binary, get_complete_input_idxs

### Adding bit-by-bit

Adding two sequences of bits A and B requires to implement a routine that sums each bit of A with the corresponding one in B. Alongside this operation, there might be carry from the immediately previous bitwise-operation (which is stored in the C).

In [2]:
def sum_bit_by_bit(A, B, C, D, n):
    addition_constaints = []
    for i in range(n+1):
        a_true = And(A[i], Not(B[i]), Not(C[i]))
        b_true = And(Not(A[i]), B[i], Not(C[i]))
        c_true = And(Not(A[i]), Not(B[i]), C[i])
        all_true = And(A[i], B[i], C[i])
        addition_constaints.append(D[i] == Or(a_true, b_true, c_true, all_true))
    return addition_constaints

### Carry

In order to generate a carry, the model checks if there are any values in A, B or C that are set as *True*.  

In [3]:
def carry(A, B, C, n):
    carry_contraints = []
    for i in range(n):
        a_b = And(A[i], B[i])
        a_c = And(A[i], C[i])
        c_b = And(C[i], B[i])
        carry_contraints.append(C[i+1] == Or(a_b, a_c, c_b))
    return carry_contraints

### Imposing input values

The final constraints the model relies on are the ones dictating the values of the inputs A and B 

In [4]:
def set_values(V, true_idxs, false_idxs):
    return [Not(V[i]) for i in false_idxs], [V[i] for i in true_idxs]

In [5]:
def variable_definition(n):
    A = [Bool(f"a_{i}") for i in range(n+1)]
    B = [Bool(f"b_{i}") for i in range(n+1)]
    
    C = [Bool(f"c_{i}") for i in range(n+1)]
    D = [Bool(f"d_{i}") for i in range(n+1)]
    
    return A,B,C,D

### SAT modelization

The last section of the exerecise about SAT encodings deals with the model creation using all the constraints so far defined. 

In [6]:
dec_a = 7
dec_b = 21

n = max(dec_a.bit_length(), dec_b.bit_length())

A,B,C,D = variable_definition(n)

solver = Solver()

solver.add(sum_bit_by_bit(A, B, C, D, n))
solver.add(carry(A, B, C, n))

solver.add(And(Not(C[0])))
solver.add(And(Not(A[n])))
solver.add(And(Not(B[n])))

A_true_idxs, A_false_idxs, B_true_idxs, B_false_idxs = decimal_to_binary(dec_a, dec_b, n)

A_true_vals, A_false_vals = set_values(A, A_true_idxs, A_false_idxs)
B_true_vals, B_false_vals = set_values(B, B_true_idxs, B_false_idxs)

solver.add(A_true_vals + A_false_vals)
solver.add(B_true_vals + B_false_vals)

if solver.check() == sat:
    m = solver.model()

    print('Binary results: ')
    print(f'\t A_bin_result: {[m[A[i]] for i in range(len(A))][::-1]}')
    print(f'\t B_bin_result: {[m[B[i]] for i in range(len(B))][::-1]}')

    print('\n')
    
    print(f'\t C_bin_result: {[m[C[i]] for i in range(len(C))][::-1]}')
    print(f'\t D_bin_result: {[m[D[i]] for i in range(len(D))][::-1]}')
    print('\n')
    print('Decimal results:')
    print(f'\t D_dec_result: {int(''.join(['1' if m[D[i]] else '0' for i in range(len(D))][::-1]),2)}')

else:
    print('Not sat')

Binary results: 
	 A_bin_result: [False, False, False, True, True, True]
	 B_bin_result: [False, True, False, True, False, True]


	 C_bin_result: [False, False, True, True, True, False]
	 D_bin_result: [False, True, True, True, False, False]


Decimal results:
	 D_dec_result: 28


## At Most One

Full adder can be employed for implementing other kind of constraints as *at most one*.

### Two Variables Example

In [7]:
n_fas = 1

x_1 = True
x_2 = True

a = [x_1]
b = [x_2]

a_idxs = [i for i,val in enumerate(a) if val]
b_idxs = [i for i,val in enumerate(b) if val]

A,B,C,D = variable_definition(n_fas)

A_true_idxs, A_false_idxs, B_true_idxs, B_false_idxs = get_complete_input_idxs(n_fas, a_idxs, b_idxs)

A_true_vals, A_false_vals = set_values(A, A_true_idxs, A_false_idxs)
B_true_vals, B_false_vals = set_values(B, B_true_idxs, B_false_idxs)

solver = Solver()

solver.add(sum_bit_by_bit(A, B, C, D, n_fas))
solver.add(carry(A, B, C, n_fas))

solver.add(And(Not(C[0]), Not(C[1])))

solver.add(A_true_vals + A_false_vals)
solver.add(B_true_vals + B_false_vals)

if solver.check() == sat:

    print('At Most One constraint satisfied!!!')

else:
    print('At Most One constraint NOT satisfied!!!')

At Most One constraint NOT satisfied!!!


### Three Variables Example

In [8]:
n_fas = 1

x_1 = True
x_2 = False
x_3 = False

a = [x_1]
b = [x_2]
c = [x_3]

a_idxs = [i for i,val in enumerate(a) if val]
b_idxs = [i for i,val in enumerate(b) if val]
c_idxs = [i for i,val in enumerate(c) if val]

A,B,C,D = variable_definition(n_fas)

A_true_idxs, A_false_idxs, B_true_idxs, B_false_idxs = get_complete_input_idxs(n_fas, a_idxs, b_idxs)
C_true_idxs, C_false_idxs, _, _ = get_complete_input_idxs(n_fas, c_idxs, [])

A_true_vals, A_false_vals = set_values(A, A_true_idxs, A_false_idxs)
B_true_vals, B_false_vals = set_values(B, B_true_idxs, B_false_idxs)
C_true_vals, C_false_vals = set_values(C, C_true_idxs, C_false_idxs)

solver = Solver()

solver.add(sum_bit_by_bit(A, B, C, D, n_fas))
solver.add(carry(A, B, C, n_fas))

solver.add(Not(C[1]))

solver.add(A_true_vals + A_false_vals)
solver.add(B_true_vals + B_false_vals)
solver.add(C_true_vals + C_false_vals)

if solver.check() == sat:
    
    print('At Most One constraint satisfied!!!')

else:
    print('At Most One constraint NOT satisfied!!!')

At Most One constraint satisfied!!!


### Four Variables Examples

In [9]:
n_fas = 2

x_1 = False
x_2 = False
x_3 = False
x_4 = True

a = [x_1, x_3]
b = [x_2, x_4]

a_idxs = [i for i,val in enumerate(a) if val]
b_idxs = [i for i,val in enumerate(b) if val]

A,B,C,D = variable_definition(n_fas)

A_true_idxs, A_false_idxs, B_true_idxs, B_false_idxs = get_complete_input_idxs(n_fas, a_idxs, b_idxs)

A_true_vals, A_false_vals = set_values(A, A_true_idxs, A_false_idxs)
B_true_vals, B_false_vals = set_values(B, B_true_idxs, B_false_idxs)

solver = Solver()

solver.add(sum_bit_by_bit(A, B, C, D, n_fas))
solver.add(carry(A, B, C, n_fas))

solver.add(And(Not(C[0]), Not(C[1]), Not(C[2]), Or(Not(D[1]), Not(D[0]))))

solver.add(A_true_vals + A_false_vals)
solver.add(B_true_vals + B_false_vals)

if solver.check() == sat:
    
    print('At Most One constraint satisfied!!!')

else:
    print('At Most One constraint NOT satisfied!!!')

At Most One constraint satisfied!!!


### Five Variables Example

In [10]:
n_fas = 2

x_1 = False
x_2 = False
x_3 = False
x_4 = False
x_5 = True

a = [x_1, x_4]
b = [x_2, x_5]
c = [x_3]

a_idxs = [i for i,val in enumerate(a) if val]
b_idxs = [i for i,val in enumerate(b) if val]
c_idxs = [i for i,val in enumerate(c) if val]

A,B,C,D = variable_definition(n_fas)

A_true_idxs, A_false_idxs, B_true_idxs, B_false_idxs = get_complete_input_idxs(n_fas, a_idxs, b_idxs)
C_true_idxs, C_false_idxs, _, _ = get_complete_input_idxs(n_fas, c_idxs, [])

A_true_vals, A_false_vals = set_values(A, A_true_idxs, A_false_idxs)
B_true_vals, B_false_vals = set_values(B, B_true_idxs, B_false_idxs)
C_true_vals, C_false_vals = set_values(C, C_true_idxs, C_false_idxs)

solver = Solver()

solver.add(sum_bit_by_bit(A, B, C, D, n_fas))
solver.add(carry(A, B, C, n_fas))

solver.add(And(Not(C[1]), Not(C[2]), Or(Not(D[1]), Not(D[0]))))

solver.add(A_true_vals + A_false_vals)
solver.add(B_true_vals + B_false_vals)
solver.add(C_true_vals + C_false_vals)

if solver.check() == sat:
    
    print('At Most One constraint satisfied!!!')

else:
    print('At Most One constraint NOT satisfied!!!')

At Most One constraint satisfied!!!


## At Most K

The full adder structure can be generalized to model an *At Most K* constraint. In the following example, K is set to be 2 using 4 variables.

In [11]:
n_fas = 2

x_1 = False
x_2 = True
x_3 = False
x_4 = True

a = [x_1, x_2]
b = [x_3, x_4]

a_idxs = [i for i,val in enumerate(a) if val]
b_idxs = [i for i,val in enumerate(b) if val]

A,B,C,D = variable_definition(n)

A_true_idxs, A_false_idxs, B_true_idxs, B_false_idxs = get_complete_input_idxs(n_fas, a_idxs, b_idxs)

A_true_vals, A_false_vals = set_values(A, A_true_idxs, A_false_idxs)
B_true_vals, B_false_vals = set_values(B, B_true_idxs, B_false_idxs)

solver = Solver()

solver.add(sum_bit_by_bit(A, B, C, D, n_fas))
solver.add(carry(A, B, C, n_fas))

amk_constr = Implies(C[2], And(Not(D[1]), Not(D[0]), Not(C[1])))
solver.add(And(Not(C[0]), amk_constr))

solver.add(A_true_vals + A_false_vals)
solver.add(B_true_vals + B_false_vals)

if solver.check() == sat:

    print('At Most Two constraint satisfied!!!')

else:
    print('At Most Two constraint NOT satisfied!!!')

At Most Two constraint satisfied!!!
